# Compare AUC using train_combined dataset only ###

In [1]:
%reset -f

In [1]:
import pandas as pd
import numpy as np

# load data
train_transaction = pd.read_csv('rawdata/train_transaction.csv')
train_identity = pd.read_csv('rawdata/train_identity.csv')
train_merged = train_transaction.merge(train_identity, on='TransactionID', how='left')
train_merged = train_merged.copy() # clear up the 'fragmentation' warning

# global feature engineering (pre-split)
train_merged['TransactionDay'] = (train_merged['TransactionDT'] / 86400).astype(int)
## ? still rounding the number huh
train_merged['D1n'] = train_merged['TransactionDay'] - train_merged['D1']

# define UID components
uid_cols = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'D1n', 'P_emaildomain']

# create UID string && handling NaNs
train_merged['UID'] = train_merged[uid_cols].fillna('NA').astype(str).agg('_'.join, axis=1)

##########
# time-based split
cutoff = train_merged['TransactionDT'].quantile(0.85)

X_train = train_merged[train_merged['TransactionDT'] <= cutoff].copy()
X_validate = train_merged[train_merged['TransactionDT'] > cutoff].copy()
##########

# calculate stats on full dataset
uid_stats = X_train.groupby('UID')['TransactionAmt'].agg(['mean', 'std', 'count']).reset_index()
uid_stats.columns = ['UID', 'UID_mean', 'UID_std', 'UID_count']
uid_stats.loc[uid_stats['UID_count'] < 3, ['UID_mean', 'UID_std']] = np.nan

# only use the UIDs that have at least 3 transactions
global_mean = X_train['TransactionAmt'].mean()

# drop UID_mean, UID_std, UID_count columns before adding them back
for df in (X_train, X_validate):
    df.drop(columns=[c for c in ['UID_mean', 'UID_std', 'UID_count'] if c in df.columns], inplace=True, errors='ignore')

# merge the manipulated stats back 
X_train = X_train.merge(uid_stats, on='UID', how='left')
X_validate = X_validate.merge(uid_stats, on='UID', how='left')

# calculate transactionAmt ratio (using global fallback for low-count UIDs)
for df in (X_train, X_validate):
    df['TransactionAmt_ratio'] = df['TransactionAmt'] / df['UID_mean'].fillna(global_mean)
    df['UID_std'] = df['UID_std'].fillna(0)
    df['UID_count'] = df['UID_count'].fillna(0)

In [2]:
# identify 'categorical' columns for lightGBM
cat_cols = X_train.select_dtypes(include=['object', 'str', 'category']).columns.tolist()
if 'UID' in cat_cols: cat_cols.remove('UID') # UID is too high cardinality

print(f"Train rows: {len(X_train):,}")
print(f"Validate rows: {len(X_validate):,}")
print(f"Categorical columns identified: {len(cat_cols)}")

Train rows: 501,959
Validate rows: 88,581
Categorical columns identified: 31


In [3]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

# features to exclude 
drop_cols = ['isFraud', 'TransactionID', 'TransactionDT', 'UID']
features = [c for c in X_train.columns if c not in drop_cols]

# intersect categorical columns with feature columns
cat_features = [c for c in cat_cols if c in features]

# convert string to 'category' type for lightGBM
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_validate[col] = X_validate[col].astype('category')

# train lightGBM
lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=256,
    random_state=42,
    n_jobs=-1,
    importance_type='gain'
)

lgbm_model.fit(
    X_train[features],
    X_train['isFraud'],
    eval_set=[(X_validate[features], X_validate['isFraud'])],
    categorical_feature=cat_features,
    eval_metric='auc',
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)]
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 17580, number of negative: 484379
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.331963 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 35861
[LightGBM] [Info] Number of data points in the train set: 501959, number of used features: 437
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035023 -> initscore=-3.316106
[LightGBM] [Info] Start training from score -3.316106
Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.912323	valid_0's binary_logloss: 0.0893936
[100]	valid_0's auc: 0.926578	valid_0's binary_logloss: 0.0833358
[150]	valid_0'

,num_leaves,256
,learning_rate,0.05
,n_estimators,500
,random_state,42
,n_jobs,-1
,importance_type,'gain'
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [4]:
# map validate categoricals onto the train categories only
# any unseen values in train becomes NaN 
# (mirror LightGBM's "unknown category -> treated as missing" behaviour)
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    train_categories = X_train[col].cat.categories

    X_validate[col] = X_validate[col].astype('category').cat.set_categories(train_categories)
    # any value in X_validate not in train_categories becomes NaN automatically

In [5]:
import xgboost as xgb
from xgboost import XGBClassifier

# XGBoost can use the same 'category' dtype columns as LightGBM
# by enable_categorical=True (requires xgboost >= 1.6, tree_method='hist')
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_dept=8,
    tree_method='hist',
    enable_categorical=True,
    eval_metric='auc',
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train[features],
    X_train['isFraud'],
    eval_set=[(X_validate[features], X_validate['isFraud'])],
    verbose=50
)

c:\Users\minim\MLcoursework\transactionfraudprediction\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [10:15:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "max_dept" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	validation_0-auc:0.77416
[50]	validation_0-auc:0.88323
[100]	validation_0-auc:0.89191
[150]	validation_0-auc:0.89844
[200]	validation_0-auc:0.90369
[250]	validation_0-auc:0.90595
[300]	validation_0-auc:0.90891
[350]	validation_0-auc:0.91066
[400]	validation_0-auc:0.91258
[450]	validation_0-auc:0.91435
[499]	validation_0-auc:0.91524


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",100
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [6]:
from catboost import CatBoostClassifier

# CatBoost prefers raw strings for categoricals, with NaNs filled explicitly
X_train_cb = X_train[features].copy()
X_validate_cb = X_validate[features].copy()

for col in cat_features:
    X_train_cb[col] = X_train_cb[col].astype(str).fillna('NA')
    X_validate_cb[col] = X_validate_cb[col].astype(str).fillna('NA')

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    eval_metric='AUC',
    random_seed=42,
    early_stopping_rounds=100,
    verbose=50
)

cat_model.fit(
    X_train_cb,
    X_train['isFraud'],
    eval_set=(X_validate_cb, X_validate['isFraud']),
    cat_features=cat_features
)

0:	test: 0.6977619	best: 0.6977619 (0)	total: 3.35s	remaining: 27m 50s
50:	test: 0.8657370	best: 0.8657370 (50)	total: 2m 33s	remaining: 22m 33s
100:	test: 0.8740118	best: 0.8740362 (99)	total: 4m 51s	remaining: 19m 12s
150:	test: 0.8801199	best: 0.8801199 (150)	total: 6m 40s	remaining: 15m 26s
200:	test: 0.8840375	best: 0.8840375 (200)	total: 8m 31s	remaining: 12m 40s
250:	test: 0.8867408	best: 0.8870408 (249)	total: 10m 14s	remaining: 10m 9s
300:	test: 0.8885397	best: 0.8885397 (300)	total: 11m 57s	remaining: 7m 54s
350:	test: 0.8910939	best: 0.8910939 (350)	total: 15m 49s	remaining: 6m 43s
400:	test: 0.8925609	best: 0.8928352 (394)	total: 18m 18s	remaining: 4m 31s
450:	test: 0.8939690	best: 0.8939690 (450)	total: 20m 44s	remaining: 2m 15s
499:	test: 0.8944538	best: 0.8947008 (479)	total: 23m 4s	remaining: 0us

bestTest = 0.8947008182
bestIteration = 479

Shrink model to first 480 iterations.


CatBoostClassifier(depth=8, early_stopping_rounds=100, eval_metric='AUC', iterations=500, learning_rate=0.05, random_seed=42, verbose=50)

In [7]:
# EVALUATION FUNCTION
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
)

preds = 0

def evaluate_model(model, X_val, y_val, model_name="Model", verbose=True):
    """
    Compare ROPC-AUC, PR-AUC, F1 (at 0.5 and at best threshold)

    Parameters
    ----------
    model: fitted classifier with predict_proba
    X_val: validation features
    y_val: true labels
    model_name: str, label for printing
    verbose: bool, whether to print results

    Return
    ----------
    dict of metrics, plus preds/thresholds for further analysis (e.g. plotting)
    """
    preds = model.predict_proba(X_val)[:, 1]

    roc_auc = roc_auc_score(y_val, preds)
    pr_auc = average_precision_score(y_val, preds)
    f1_default = f1_score(y_val, (preds >= 0.5).astype(int))

    precision, recall, thresholds = precision_recall_curve(y_val, preds)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-12)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 1.0
    f1_best = f1_scores[best_idx]

    results = {
        "model_name": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "f1_default": f1_default,
        "f1_best": f1_best,
        "best_threshold": best_threshold,
        "preds": preds
    }

    if verbose:
        print(f"--- {model_name} ---")
        print(f"ROC-AUC:              {roc_auc:.5f}")
        print(f"PR-AUC:               {pr_auc:.5f}")
        print(f"F1 @ 0.5 threshold:   {f1_default:.5f}")
        print(f"F1 @ best threshold ({best_threshold:.4f}): {f1_best:.5f}")
        print()

    return results

In [8]:
lgbm_result = evaluate_model(lgbm_model, X_validate[features], X_validate['isFraud'], model_name="LightGBM")
xgb_result = evaluate_model(xgb_model, X_validate[features], X_validate['isFraud'], model_name="XGBoost")#
cat_result = evaluate_model(cat_model, X_validate_cb, X_validate['isFraud'], model_name="CatBoost")

--- LightGBM ---
ROC-AUC:              0.93313
PR-AUC:               0.62174
F1 @ 0.5 threshold:   0.54274
F1 @ best threshold (0.2532): 0.59546

--- XGBoost ---
ROC-AUC:              0.91524
PR-AUC:               0.58648
F1 @ 0.5 threshold:   0.51294
F1 @ best threshold (0.2419): 0.56288

--- CatBoost ---
ROC-AUC:              0.89470
PR-AUC:               0.54947
F1 @ 0.5 threshold:   0.49011
F1 @ best threshold (0.2544): 0.55293



In [9]:
# COMPARING different models
all_results = [lgbm_result, xgb_result, cat_result] # append more as you train other models
comparison_df = pd.DataFrame(all_results)[['model_name', 'roc_auc', 'pr_auc', 'f1_default', 'f1_best', 'best_threshold']]
comparison_df

,model_name,roc_auc,pr_auc,f1_default,f1_best,best_threshold
0,LightGBM,0.933131,0.621739,0.542744,0.595457,0.253247
1,XGBoost,0.915244,0.586479,0.512936,0.562877,0.241911
2,CatBoost,0.894701,0.549470,0.490106,0.552929,0.254416


In [10]:
import json
from pathlib import Path

# default operating threshold: use best threshold from lightGBM = 0.253
# can be changed depending on business decision
DEFAULT_THRESHOLD = 0.253

# ---------------------
# export training artifacts
# ---------------------
def export_training_artifacts(model, uid_stats, global_mean, cat_features, X_train, features, model_type, out_dir='artifacts'):
    # Call this function once after training the models to persist the model artifacts
    # model_type: 'lightgbm'|'xgboost'|'catboost'
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1. UID stats and global mean
    uid_stats.to_pickle(out_dir / 'uid_stats.pkl')
    
    with open(out_dir / 'global_mean.json', 'w') as f:
        json.dump({'global_mean': float(global_mean)}, f)

    # 2. features list and categorical features list (order matters!)
    with open(out_dir / 'feature_config.json', 'w') as f:
        json.dump({
            'features': list(features),
            'cat_features': list(cat_features),
            'model_type': model_type,
            'threshold': DEFAULT_THRESHOLD,
        }, f, indent=2)

    # 3. trained category vocabularies (there could be unseen category value during inference)
    train_categories = {
        col: list(X_train[col].astype('category').cat.categories)
        for col in cat_features
    }
    with open(out_dir / 'train_categories.json', 'w') as f:
        json.dump(train_categories, f)

    # 4. the models
    if model_type == 'lightgbm':
        model.booster_.save_model(str(out_dir / 'model_lightgbm.txt'))
    elif model_type == 'xgboost':
        model.save_model(str(out_dir / 'model_xgboost.json'))
    elif model_type == 'catboost':
        model.save_model(str(out_dir / 'model_catboost.cbm'))
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    print(f"Artifacts exported to {out_dir.resolve()}")

In [11]:
export_training_artifacts(lgbm_model, uid_stats, global_mean, cat_features, X_train, features, 'lightgbm')

Artifacts exported to C:\Users\minim\MLcoursework\transactionfraudprediction\artifacts


In [12]:
train_merged.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,TransactionDay,D1n,UID
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,-13.0,13926_NA_150.0_discover_142.0_credit_315.0_87....
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0,2755_404.0_150.0_mastercard_102.0_credit_325.0...
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0,4663_490.0_150.0_visa_166.0_debit_330.0_87.0_1...
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,-111.0,18132_567.0_150.0_mastercard_117.0_debit_476.0...
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,1,1.0,4497_514.0_150.0_mastercard_102.0_credit_420.0...


In [13]:
test_transaction = pd.read_csv('rawdata/test_transaction.csv')
test_identity = pd.read_csv('rawdata/test_identity.csv')
test_merged = test_transaction.merge(test_identity, on='TransactionID', how='left')
test_merged = test_merged.copy() # clear up the 'fragmentation' warning

test_merged.head()

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
lgbm_result['preds']

array([0.00162537, 0.00237057, 0.00553471, ..., 0.00153392, 0.02797355,
       0.00474665], shape=(88581,))

In [15]:
import sys
sys.path.append('.')  # location of fraud_inference.py
from test_fraud_inference import run_consistency_check

# X_validate_raw should be the raw slice — grab it fresh before any
# UID/category engineering, e.g. from train_merged using the same time split
X_validate_raw = train_merged[train_merged['TransactionDT'] > cutoff].copy()

run_consistency_check(
    X_validate_raw=X_validate_raw, # the raw dataframe before any manipuation (with isFraud column)
    notebook_preds=lgbm_result['preds'],   # predict_proba output already computed in the previous cell
    artifact_dir='artifacts',
    n_sample=2000,  # or None to check all rows
)

Checked 2,000 rows
Max abs diff:  0.00e+00
Mean abs diff: 0.00e+00
Rows beyond tolerance (1e-06): 0

✅ PASSED — fraud_inference.py reproduces notebook predictions.


{'n_rows_checked': 2000,
 'max_abs_diff': 0.0,
 'mean_abs_diff': 0.0,
 'n_rows_beyond_tolerance': 0,
 'tolerance': 1e-06,
 'passed': True}

In [16]:
# sanity check
# grab 100 random rows from the validateion set
test_sample = X_validate.sample(100, random_state=42)
test_sample.to_csv('sanity_check_input.csv', index=False)

# get the True predictions
true_probs = lgbm_model.predict_proba(test_sample[features])[:,1]

# instructions for user
print("1. Run this command in your terminal:")
print("python fraud_inference.py --input sanity_check_input.csv --output sanity_check_output.csv")
print("\n2. Load the output and compare:")
print("output_df = pd.read_csv('sanity_check_output.csv')")
print("print(np.isclose(true_probs, output_df['Fraud_Probability'], rtol=1e-4).all())")

1. Run this command in your terminal:
python fraud_inference.py --input sanity_check_input.csv --output sanity_check_output.csv

2. Load the output and compare:
output_df = pd.read_csv('sanity_check_output.csv')
print(np.isclose(true_probs, output_df['Fraud_Probability'], rtol=1e-4).all())


In [18]:
output_df = pd.read_csv('sanity_check_output.csv')
print(np.isclose(true_probs, output_df['fraud_probability'], rtol=1e-4).all())

True
